# 3.2.1 图像识别评估系统交互流程设计

## 任务说明（摘自考试预览）

图像识别评估系统是在深度学习技术日益成熟的背景下发展起来的，旨在解决传统图像识别方法在面对复杂场景和大规模数据集时的局限性。随着互联网和物联网技术的飞速发展，图像数据量呈指数级增长，对图像内容的自动理解和智能分析提出了更高的要求。ResNet作为一种深度卷积神经网络架构，凭借其深度残差连接机制，能够有效缓解梯度消失问题，实现更深层次的网络结构，从而捕获更加丰富和抽象的图像特征，极大地提高了图像识别的准确性和效率，推动了人工智能技术在现实世界中的广泛应用和商业化进程。

AI模型说明：“resnet.onnx”模型是使用 Pytorch 框架和基于深度卷积神经网络网络训练得到的，专门用于进行图像识别。对应的标签文件为“labels.txt”。该模型的使用交互流程为：

1)加载“resnet.onnx”模型和“labels.txt”类别标签；

2)加载本地测试图片“img_test.jpg”，并进行预处理图像以符合模型输入要求；

3)使用“resnet.onnx”模型对加载的图片进行识别；

4)输出加载图片的识别结果（输出概率值最大的5组类别和对应概率值）

你作为一名人工智能训练师，请完成以下工作任务：

（1）补全该模型的使用交互流程对应的Python代码（3.2.1.ipynb），实现本地测试图片“img_test.jpg”的识别，将其识别结果截图保存为jpg格式文件，命名为3.2.1-1.jpg。

（2）在上面的使用交互流程基础上，给出在图像识别评估系统中使用“resnet.onnx”模型的一种人机交互的最优方式，将其保存为docx文件，命名为3.2.1.docx。

所有结果文件储存在桌面新建的考生文件夹中，文件夹命名为“准考证号+身份证号后六位”。

---

以下为**刷题参考模板**：含完整可运行代码（编程题）或答题示例与生成 docx 草稿代码（主观题）。

## ONNX 图像分类 Top5

In [ ]:
import onnxruntime as ort
import numpy as np
import scipy.special
from PIL import Image


def preprocess_image(
    image,
    resize_size=256,
    crop_size=224,
    mean=(0.485, 0.456, 0.406),
    std=(0.229, 0.224, 0.225),
):
    image = image.resize((resize_size, resize_size), Image.BILINEAR)
    w, h = image.size
    left = (w - crop_size) / 2
    top = (h - crop_size) / 2
    image = image.crop((left, top, left + crop_size, top + crop_size))
    image = np.array(image).astype(np.float32)
    image = image / 255.0
    image = (image - mean) / std
    image = np.transpose(image, (2, 0, 1))
    image = image.reshape((1,) + image.shape)
    return image


session = ort.InferenceSession(
    'ResNet50.onnx', providers=['CPUExecutionProvider']
)

with open('labels.txt', encoding='utf-8') as f:
    labels = [line.strip() for line in f.readlines()]

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

image = Image.open('demo.jpg').convert('RGB')
processed_image = preprocess_image(image)
processed_image = processed_image.astype(np.float32)

output = session.run(
    [output_name], {input_name: processed_image}
)[0]

probabilities = scipy.special.softmax(output, axis=-1)[0]
top5_idx = np.argsort(probabilities)[-5:][::-1]
top5_prob = probabilities[top5_idx]

print("Top 5 predicted classes:")
for i in range(5):
    print(
        f'{i+1}: {labels[top5_idx[i]]} - Probability: {top5_prob[i]}'
    )

## 学习提示

- 编程题：请在**本案例文件夹**下运行 Notebook，以便相对路径 `data/...` 正确。
- 主观题：示例答案仅供参考，可按考场要求改写；若未安装 `python-docx`，可先 `pip install python-docx`。
- 交卷前核对平台要求的截图、HTML、docx 命名与保存路径。